# Transformer Accounting

## Basic Memory Calculations
Let $E$ be embedding mem, $L$ num layers
$$ 
M = \underbrace{|Vocab|d}_{E} + L\left(\underbrace{3d^2 + d^2}_{W_{QKV} + W_O} + \underbrace{3d_{ff}d}_{FFN} + 2\underbrace{d}_{\text{LN}}\right) + \underbrace{d}_{\text{LN}} + \underbrace{|Vocab|d}_{\text{Ouput}}
$$


## Basic FLOPS Calculations

Transformer block dominates, focus there. Split into MHA and FFN. Let $c$ be context length, $h$ be number of heads
$$
MHA = \underbrace{6cd^2}_{Q, K, V} + h\left(\underbrace{2c^2(d/h)}_{Q_hK_h^T} + \underbrace{2c^2(d/h)}_{\text{softmax}\cdot V_h}\right) + \underbrace{2d^2c}_{W_O \text{ mult}}
$$
All the multiplies take same amount of time.
$$
FFN = \underbrace{6d_{ff} d c}_{W_2(\sigma(W_1x)\odot W_3x)}
$$

### Flops Conclusion

When context lenght dominates, MHA is more expensive. When $d_{ff}$ dominates, the FFN is more expensive.

In [20]:

def nb_trainable_params(param_dict):
    # Embedding layer

    vocab_size = param_dict['vocab_size']
    num_layers = param_dict['num_layers']
    d_model = param_dict['d_model']
    d_ff = param_dict['d_ff']

    nb_params = 0
    embedding = d_model*vocab_size
    nb_params += embedding
    # Transformer Block layers
    ## Feed forward layer
    ffn_w1 = d_ff*d_model
    ffn_w2 = d_ff*d_model
    ffn_w3 = d_model * d_ff
    nb_params += ffn_w1+ffn_w2+ ffn_w3

    ## MHA params
    W_QKV = 3*d_model**2
    W_O = d_model**2
    nb_params += W_QKV + W_O

    ## Layer Norm 
    ln_in_block = 2*d_model
    nb_params += ln_in_block

    ## Multiplied by num_layers 
    nb_params *= num_layers

    # final LN
    ln_final = d_model
    nb_params += ln_final

    # final output layer
    output = vocab_size * d_model
    nb_params += output

    return nb_params

# flops focusing on costly parts
def nb_flops(param_dict): 
    nb_flops = 0
    nb_flops_by_part = {'MHA' : 0, 'FFN' : 0, 'ouput' : 0} 
    vocab_size = param_dict['vocab_size']
    num_layers = param_dict['num_layers']
    context_length = param_dict['context_length']
    d_model = param_dict['d_model']
    num_heads = param_dict['num_heads']
    d_ff = param_dict['d_ff']
    
    # Transformer block layers
    
    ## MHA (ignoring softmax)
    ### Project input to weights QKV
    proj_flops = 2*(3*d_model**2)*(context_length)
    ### Per heads QK^T
    QKT_flops = 2*(context_length**2)*(d_model//num_heads)
    ### Value flops
    V_flops = 2*(context_length**2)*(d_model//num_heads)
    ### W_O flops
    W_O_flops  = 2*d_model**2 * context_length
    nb_flops_by_part['MHA'] = proj_flops+ num_heads*(QKT_flops+V_flops)+W_O_flops
    nb_flops_by_part['MHA'] *= num_layers
    
    ## FFN
    ### Multiply input (ignoring nonlinearity and the odot) 
    W1x_flops = 2*d_ff*d_model*context_length
    W3x_flops = W1x_flops
    W2_sigma_W1x_dot_W3x = W1x_flops
    nb_flops_by_part['FFN'] = W1x_flops + W3x_flops + W2_sigma_W1x_dot_W3x
    nb_flops_by_part['FFN'] *= num_layers

    ## Output layer
    nb_flops_by_part['output'] = 2*vocab_size*d_model*context_length

    for count in nb_flops_by_part.values():
        nb_flops += count
    return nb_flops, nb_flops_by_part



## GPT-2-XL

In [21]:
# GPT-2 XL 
gpt_2_xl = {}
gpt_2_xl['vocab_size'] = 50257
gpt_2_xl['context_length'] = 1024
gpt_2_xl['num_layers'] = 48
gpt_2_xl['d_model'] = 1600
gpt_2_xl['num_heads'] = 25
gpt_2_xl['d_ff'] = 6400

nb_trainable = nb_trainable_params(gpt_2_xl)
print("Number of trainable parameters", nb_trainable)

print("Memory consumed", nb_trainable*4 /(10**9), "GBs")

nb_flops_total, nb_flops_by_part = nb_flops(gpt_2_xl)
print("Number of flops total", nb_flops_total/(10**12), "Teraflops")
print("Number of MHA flops", nb_flops_by_part['MHA']/(10**12), "Teraflops")
print("Number of FFN flops", nb_flops_by_part['FFN']/(10**12), "Teraflops")
print("Number of output flops", nb_flops_by_part['output']/(10**12), "Teraflops")

print("FFN/MHA", nb_flops_by_part['FFN']/nb_flops_by_part['MHA'])

Number of trainable parameters 5906384000
Memory consumed 23.625536 GBs
Number of flops total 4.5133365248 Teraflops
Number of MHA flops 1.3287555072 Teraflops
Number of FFN flops 3.01989888 Teraflops
Number of output flops 0.1646821376 Teraflops
FFN/MHA 2.272727272727273


## GPT-2 Small

In [22]:
# GPT-2 XL 
gpt_2_small = {}
gpt_2_small['vocab_size'] = 50257
gpt_2_small['context_length'] = 1024
gpt_2_small['num_layers'] = 12
gpt_2_small['d_model'] = 768
gpt_2_small['num_heads'] = 12
gpt_2_small['d_ff'] = 3072

nb_trainable = nb_trainable_params(gpt_2_small)
print("Number of trainable parameters", nb_trainable)

print("Memory consumed", nb_trainable*4 /(10**9), "GBs")

nb_flops_total, nb_flops_by_part = nb_flops(gpt_2_small)
print("Number of flops total", nb_flops_total/(10**12), "Teraflops")
print("Number of MHA flops", nb_flops_by_part['MHA']/(10**12), "Teraflops")
print("Number of FFN flops", nb_flops_by_part['FFN']/(10**12), "Teraflops")
print("Number of output flops", nb_flops_by_part['output']/(10**12), "Teraflops")

print("FFN/MHA", nb_flops_by_part['FFN']/nb_flops_by_part['MHA'])

Number of trainable parameters 615031296
Memory consumed 2.460125184 GBs
Number of flops total 0.349630365696 Teraflops
Number of MHA flops 0.09663676416 Teraflops
Number of FFN flops 0.173946175488 Teraflops
Number of output flops 0.079047426048 Teraflops
FFN/MHA 1.8


## GPT-2 Large

In [23]:
# GPT-2 XL 
gpt_2_large = {}
gpt_2_large['vocab_size'] = 50257
gpt_2_large['context_length'] = 1024
gpt_2_large['num_layers'] = 24
gpt_2_large['d_model'] = 1024
gpt_2_large['num_heads'] = 16
gpt_2_large['d_ff'] = 5120

nb_trainable = nb_trainable_params(gpt_2_large)
print("Number of trainable parameters", nb_trainable)

print("Memory consumed", nb_trainable*4 /(10**9), "GBs")

nb_flops_total, nb_flops_by_part = nb_flops(gpt_2_large)
print("Number of flops total", nb_flops_total/(10**12), "Teraflops")
print("Number of MHA flops", nb_flops_by_part['MHA']/(10**12), "Teraflops")
print("Number of FFN flops", nb_flops_by_part['FFN']/(10**12), "Teraflops")
print("Number of output flops", nb_flops_by_part['output']/(10**12), "Teraflops")

print("FFN/MHA", nb_flops_by_part['FFN']/nb_flops_by_part['MHA'])

Number of trainable parameters 1764780032
Memory consumed 7.059120128 GBs
Number of flops total 1.187728326656 Teraflops
Number of MHA flops 0.309237645312 Teraflops
Number of FFN flops 0.77309411328 Teraflops
Number of output flops 0.105396568064 Teraflops
FFN/MHA 2.5


# GPT-XL long context

In [24]:
# GPT-2 XL 
gpt_2_xl = {}
gpt_2_xl['vocab_size'] = 50257
gpt_2_xl['context_length'] = 16384
gpt_2_xl['num_layers'] = 48
gpt_2_xl['d_model'] = 1600
gpt_2_xl['num_heads'] = 25
gpt_2_xl['d_ff'] = 6400

nb_trainable = nb_trainable_params(gpt_2_xl)
print("Number of trainable parameters", nb_trainable)

print("Memory consumed", nb_trainable*4 /(10**9), "GBs")

nb_flops_total, nb_flops_by_part = nb_flops(gpt_2_xl)
print("Number of flops total", nb_flops_total/(10**12), "Teraflops")
print("Number of MHA flops", nb_flops_by_part['MHA']/(10**12), "Teraflops")
print("Number of FFN flops", nb_flops_by_part['FFN']/(10**12), "Teraflops")
print("Number of output flops", nb_flops_by_part['output']/(10**12), "Teraflops")

print("FFN/MHA", nb_flops_by_part['FFN']/nb_flops_by_part['MHA'])

Number of trainable parameters 5906384000
Memory consumed 23.625536 GBs
Number of flops total 149.5227957248 Teraflops
Number of MHA flops 98.5694994432 Teraflops
Number of FFN flops 48.31838208 Teraflops
Number of output flops 2.6349142016 Teraflops
FFN/MHA 0.49019607843137253


# AdamW Accounting (page 32)
- Assume that $d_{ff} = 4d$ if you want.
- Let $b$ denote batch size.

## Params
$$ 
M = \underbrace{|Vocab|d}_{E} + L\left(\underbrace{3d^2 + d^2}_{W_{QKV} + W_O} + \underbrace{3d_{ff}d}_{FFN} + 2\underbrace{d}_{\text{LN}}\right) + \underbrace{d}_{\text{LN}} + \underbrace{|Vocab|d}_{\text{Ouput}}
$$

## Optimizer states

This requires 2*M

## Activations

This isn't defined in the assignment (maybe it's in a lecture?). "Activations" are tensors you compute in the forward pass that are 'needed' to compute the jacobian in the backward pass. ('needed' is somewhat fuzzy because it depends on how you implement autograd). So if you have
$$
f(g(h(x)) \implies Df(g(h(x)))Dg(h(x))Dh(x)
$$
the activations are $x$, $h(x)$, and $g(h(x))$. These are all needed in the backward pass. 

We also sometimes refer to the activations belonging to a sub operation, e.g., $g$ in the above composition. We can think of the activations belonging to $g$ as those tensors that are created when evaluating $g$ and which are 'needed' to compute the final backward pass. Sometimes a parameter is needed in computing the backward pass. We usually do not count that as an activation since it's already stored.

For a transformer, the input is a batch of size $b$ of data points (sequences of length $c$). We can think of this as a tensor $X \in \mathbb{R}^{b \times c \times d}$ (already embedded).
- RMSNorms: The RMSNorm is $\gamma \odot \hat X$ where $\hat X$ is a "normalized" input (across the $d$ dimension.). In the backward pass, to compute the jacobian with respect to $\gamma$, we just need this normalized input. To compute the jacobian with respect to $X$ we need $\gamma$, the column normalized $\hat X$ and $r$, which is the denominator `mean(x**2) + eps`. $\gamma$ is already stored because it is a parameter. Thus we do not count it as activation memory. Therefore, the total activation memory is:
    - Memory: $bcd + bc$ (often the $bc$ part is ignored).
- $Q,K,V$ projections: This is a matrix vector product: $[Q, K, V] = X^TW_{QKV}$, etc. So Jacobian with respect to $X$ needs $W_{QKV}$ (already stored) and the jacobian with respect to $W_{QKV}$ just needs $X$. Note that later in the network, we will need $Q, K, V$, so we must also store those. Thus the number is 
    - Memory: $4bcd$.
- $Q^TK$ matrix multiply: to compute the jacobian, we just need $Q$ and $K$, which are already stored. But we will later on need $Q^TK$, so we store that. (these are called 'logits').
    - Memory: $bhc^2$
        - note this is huge and grows with number of heads.
- Softmax: Consider just the softmax of vector input $f(z) = softmax(z)$. Then $\nabla f(x) = \text{diag}(s) - ss^T$ where $s = softmax(z)$. thus, to compute the jacobian, we just need the softmax. This means that the total memory incurred is: 
    - Memory: $bhc^2$
        - note that we can discard the logits from the previous step here. They are no longer needed.
- Weighted some of values: we need to compute $s*V_h$ where $s$ are the incoming soft max and $V_h$ is the head. We're already storing $s$ and $V_h$, so we just save the output which is 
    - $b c d$ dimensional. 
- Output projection: we need to multiply by $W_O$. The Jacobian here will only involve $W_O$ and the input. We're already saving the input, so we just need the output which is needed downstream. Thus we just save: 
    - Memory: $bcd$.
- Feedforward memory: We need to compute $U = W_1 X$ and $U' = W_3 X$ (both $bcd_ff = 4bcd$), we also need to compute $Z = SiLU(U) \odot U'$ which is same memory. Then we need $W_1Z$ which is $bcd$. So we need 
    - memory: $13bcd$.
- FinalRMS: Same as above. 
    - Memory: $bcd + bc$
- Output embedding: Assuming input is saved, just need to save the output (logits) which is size 
    - Memory: $bc|Vocab|$
- Cross entropy: the loss is $f(logits) = \text{avg}_{i, batch}(-log(softmax(logits)[x_{i+1}]))$. The gradient requires the targets and softmax(logits) which can be recomputed on the fly given logits, so no new memory is incurred.

Total: $bc|Vocab| + L(bhc^2 + (2 + 4 + 1 + 1 + 13)bcd) + bcd$


# Peak Memory of params, states, and activations
$$
3\left(\underbrace{|Vocab|d}_{E} + L\left(\underbrace{3d^2 + d^2}_{W_{QKV} + W_O} + \underbrace{12d^2}_{FFN} + 2\underbrace{d}_{\text{LN}}\right) + \underbrace{d}_{\text{LN}} + \underbrace{|Vocab|d}_{\text{Ouput}}\right) + bc|Vocab| + L(bhc^2 + 20bcd) + bcd
$$

In [25]:
import math
def peak_memory(param_dict):
    vocab_size = param_dict['vocab_size']
    num_layers = param_dict['num_layers']
    context_length = param_dict['context_length']
    d_model = param_dict['d_model']
    num_heads = param_dict['num_heads']
    d_ff = param_dict['d_ff']

    total_memory_params_states = 3*(vocab_size*d_model + num_layers*(16*d_model**2 + 2*d_model) + 2*d_model) + d_model + vocab_size*d_model
    total_memory_params_states = 4*total_memory_params_states/10**9
    total_memory_activations = context_length*vocab_size + num_layers* (num_heads*context_length**2 + 20*context_length*d_model) + context_length*d_model
    total_memory_activations = 4*total_memory_activations/10**9

    print(f"Total memory batch_size*{total_memory_activations} + {total_memory_params_states} GBs")
    print("Max batch size so that memory is under 80GBs:", math.floor((80 - total_memory_params_states)/total_memory_activations))


In [26]:
# GPT-2 XL 
gpt_2_xl = {}
gpt_2_xl['vocab_size'] = 50257
gpt_2_xl['context_length'] = 1024
gpt_2_xl['num_layers'] = 48
gpt_2_xl['d_model'] = 1600
gpt_2_xl['num_heads'] = 25
gpt_2_xl['d_ff'] = 6400
peak_memory(gpt_2_xl)


Total memory batch_size*11.537027072 + 24.8814272 GBs
Max batch size so that memory is under 80GBs: 4


# Number of Flops for one step of AdamW

- Forward pass same as evaluating the model. We'll say the backward pass takes twice times as many flops as backward pass. 
- Then we need to do the step

In [27]:
# GPT-2 XL 
gpt_2_xl = {}
gpt_2_xl['vocab_size'] = 50257
gpt_2_xl['context_length'] = 1024
gpt_2_xl['num_layers'] = 48
gpt_2_xl['d_model'] = 1600
gpt_2_xl['num_heads'] = 25
gpt_2_xl['d_ff'] = 6400

nb_flops_forward_pass, _ = nb_flops(gpt_2_xl)
nb_trainable = nb_trainable_params(gpt_2_xl)

forward_plus_backward_flops = 3*nb_flops_forward_pass
update_m = 3*nb_trainable # multiply two big params by scalar then average.
update_v = update_m # same sort of thing, except we just square entries of g.
update_theta = (1+1+1+1+1)*nb_trainable # sqrt + add eps + divide + mult by alpha + subtract.
update_theta += (1+1)*nb_trainable # scale by alpha lambda, then subtract.

total_flops = (forward_plus_backward_flops + update_m + update_v + update_theta)
print(f"Total Flops for step of AdamW {total_flops/(10**12)}*batch_size TFlOPs")

MFU = .5
A100tflops_per_s = 19.5
nb_steps = 4*10**5
batch_size = 1024

seconds_per_step = total_flops*batch_size/(MFU*A100tflops_per_s)/(10**12) # thanks to @wowitsmrinal for noting the missing 'batch_size' in the original.
print(f"Seconds per step: {seconds_per_step}")
print(f"To perform {nb_steps} steps with batch size {batch_size} it would take {nb_steps*seconds_per_step/60/60/24/365} years")


Total Flops for step of AdamW 13.6167925664*batch_size TFlOPs
Seconds per step: 1430.1123679993434
To perform 400000 steps with batch size 1024 it would take 18.139426281067266 years
